# اليوم الثالث — مختبر 6: التقييم وتحليل الأخطاء
## Day 3 — Lab 6: Evaluation & Error Analysis

**الطالبة**: منال قيسي


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Manal-qaysi/bayan-nlp-Manal-qaysi/blob/main/notebooks/07_evaluation_error_analysis.ipynb)

**الهدف:** الانتقال من رقم عام إلى دليل: Macro-F1 + bootstrap CI + paired comparison + slices + behavioural tests + taxonomy + إصلاحات مرتبة.

**Goal:** produce an honest evaluation report that shows uncertainty and actionable failure modes.


> 🟣 **نسخة مشروع منال قيسي (Manal-qaysi).** المختبر الأصلي من مادة الدورة، والخلايا المعنونة بـ«🟣 مشروعي» أُضيفت لمشروعي: تربط الدفتر بمستودعي وبياناته وتطبع نتائج قابلة للتتبع بصيغة `BAYAN_RESULT::…` يجمعها `scripts/collect_results.py`.
> Run with **Runtime → Restart session and run all**, then **File → Save a copy in GitHub** to `notebooks/07_evaluation_error_analysis.ipynb` in `Manal-qaysi/bayan-nlp-Manal-qaysi`.

## مهم: ما مصدر التنبؤات؟ | Evidence provenance

> 🟣 الأقسام 1–7 تمرين الدورة على `COURSE_FIXTURE` وتبقى موسومة كذلك. **تقييم مشروعي الفعلي** في قسم «🟣 مشروعي» بعدها: يقرأ تنبؤات نماذجي من مخرجات دفاتري 03–06 المحفوظة في المستودع.

ملف `bayan_day3_predictions.csv` هو `COURSE_FIXTURE`: تنبؤات تعليمية مكتوبة خصيصًا لتعلم القياس، وليست نتائج نموذج مخفي ولا benchmark. عند استخدام مشروعك، استبدلها بتنبؤات validation فعلية مع الحفاظ على الأعمدة.

لا نستخدم frozen test لصناعة taxonomy أو اختيار الإصلاح؛ نستخدم validation حتى يظل test تقييمًا نهائيًا.

In [1]:
import importlib.metadata
import subprocess
import sys

REQUIRED = {"scikit-learn": "1.9.0"}
try:
    current = importlib.metadata.version("scikit-learn")
except importlib.metadata.PackageNotFoundError:
    current = None
if current != REQUIRED["scikit-learn"]:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "scikit-learn==1.9.0"
    ])
assert importlib.metadata.version("scikit-learn") == "1.9.0"
print("SETUP=PASS", importlib.metadata.version("scikit-learn"))


SETUP=PASS 1.9.0


In [2]:
# 🟣 مشروعي — ربط الدفتر بمستودعي: نسخة الكود والبيانات وبصماتها (SHA-256)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Manal-qaysi/bayan-nlp-Manal-qaysi"
_candidates = [Path.cwd(), Path.cwd().parent, Path("/content/bayan-nlp-Manal-qaysi")]
REPO_DIR = next((p for p in _candidates if (p / "src" / "bayan" / "project.py").is_file()), None)
if REPO_DIR is None:
    REPO_DIR = Path("/content/bayan-nlp-Manal-qaysi") if Path("/content").is_dir() else Path("bayan-nlp-Manal-qaysi").resolve()
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from bayan.project import data_hashes, emit_result, git_head

DATA_DIR = REPO_DIR / "data" / "sample"
REPO_SHA = git_head(REPO_DIR)
DATA_SHA256 = data_hashes(DATA_DIR)
PROVENANCE = {"repo": REPO_URL, "repo_sha": REPO_SHA}
print("REPO_DIR:", REPO_DIR)
print("REPO_SHA:", REPO_SHA)
for _name, _digest in DATA_SHA256.items():
    print(f"  {_name}  sha256={_digest[:16]}…")

REPO_DIR: /content/bayan-nlp-Manal-qaysi
REPO_SHA: 03e1f481cd6c47b570296b0dde7a46b1799f5a9c
  bayan_day1_sample.csv  sha256=904a5e1e860f23ac…
  bayan_day2_classification.csv  sha256=c50de92fdab1aa36…
  bayan_day2_ner.jsonl  sha256=ab413f0941656abf…
  bayan_day2_qa.json  sha256=4e894757b74d09df…
  bayan_day3_arabic.csv  sha256=0a3346b6177d0c0b…
  bayan_day3_cases.csv  sha256=322867b54d1f6f35…
  bayan_day3_predictions.csv  sha256=63b4df9dab076880…
  bayan_day3_queries.jsonl  sha256=f80ebd8b25c37b33…


In [3]:
import csv
import io
import json
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
from sklearn.metrics import f1_score

DATA_KIND = "COURSE_FIXTURE"
print("IMPORTS=PASS")


IMPORTS=PASS


In [4]:
DATA_URL = (DATA_DIR / "bayan_day3_predictions.csv").as_uri()
FALLBACK_ROWS = [{'example_id': 'EV-001', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'permit', 'prediction_b': 'digital_service', 'text': 'ما وصلني رمز التحقق'}, {'example_id': 'EV-002', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'health', 'text': 'أبي أغير موعد العيادة لأن الوقت الحالي ما يناسبني'}, {'example_id': 'EV-003', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'permit', 'prediction_b': 'permit', 'text': 'طلبي واقف بالمراجعة'}, {'example_id': 'EV-004', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'الباص تأخر علينا أكثر من ساعة وما وصل إشعار'}, {'example_id': 'EV-005', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'التطبيق يعلق عند الدخول'}, {'example_id': 'EV-006', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'النتيجة للحين ما ظهرت بالتطبيق رغم مرور يومين'}, {'example_id': 'EV-007', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'المرفق انرفض'}, {'example_id': 'EV-008', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'المسار الجديد مو موجود في الخريطة بعد التحديث'}, {'example_id': 'EV-009', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'الكود ما وصل'}, {'example_id': 'EV-010', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'digital_service', 'text': 'أحتاج أجدد الوصفة من التطبيق لكن الزر ما يشتغل'}, {'example_id': 'EV-011', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'permit', 'prediction_b': 'permit', 'text': 'انخصمت الرسوم مرتين'}, {'example_id': 'EV-012', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'وين موقف الباص الجديد في الحي؟'}, {'example_id': 'EV-013', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'تعذر الدخول إلى البوابة'}, {'example_id': 'EV-014', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'health', 'text': 'لا تتوفر مواعيد مناسبة في العيادة المطلوبة'}, {'example_id': 'EV-015', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'permit', 'prediction_b': 'permit', 'text': 'رُفض المستند المرفق'}, {'example_id': 'EV-016', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'تأخرت الحافلة عن الموعد المحدد بنصف ساعة'}, {'example_id': 'EV-017', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'permit', 'prediction_b': 'digital_service', 'text': 'يفشل رفع الملف'}, {'example_id': 'EV-018', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'health', 'text': 'لم تظهر نتيجة الفحص في الملف الصحي الإلكتروني'}, {'example_id': 'EV-019', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'حالة التصريح معلقة'}, {'example_id': 'EV-020', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'digital_service', 'text': 'لا يظهر مسار الحافلة في التطبيق'}, {'example_id': 'EV-021', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'لم يصل رمز التحقق'}, {'example_id': 'EV-022', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'health', 'text': 'تعذر تجديد الوصفة الطبية عبر الخدمة الإلكترونية'}, {'example_id': 'EV-023', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'permit', 'prediction_b': 'permit', 'text': 'تم خصم الرسم مرتين'}, {'example_id': 'EV-024', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'أرغب في معرفة موقع أقرب محطة للحافلات'}, {'example_id': 'EV-025', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'The sign in code did not arrive'}, {'example_id': 'EV-026', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'health', 'text': 'I need to move my clinic appointment to another day'}, {'example_id': 'EV-027', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'permit', 'prediction_b': 'permit', 'text': 'My permit document was rejected'}, {'example_id': 'EV-028', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'The bus arrived forty minutes after the scheduled time'}, {'example_id': 'EV-029', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'The portal freezes'}, {'example_id': 'EV-030', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'My laboratory result is still missing from the health record'}, {'example_id': 'EV-031', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'The request is stuck'}, {'example_id': 'EV-032', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'The new bus route is missing from the mobile map'}, {'example_id': 'EV-033', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'PDF upload failed'}, {'example_id': 'EV-034', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'health', 'text': 'The prescription renewal button does not work in the application'}, {'example_id': 'EV-035', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'permit', 'prediction_b': 'permit', 'text': 'I was charged twice'}, {'example_id': 'EV-036', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'I cannot find the neighbourhood bus stop'}]

try:
    with urllib.request.urlopen(DATA_URL, timeout=15) as response:
        rows = list(csv.DictReader(io.StringIO(response.read().decode("utf-8"))))
    data_source = "github"
except Exception as exc:
    rows = FALLBACK_ROWS
    data_source = f"embedded_fallback:{type(exc).__name__}"

required = {"example_id", "split", "language", "variant", "length_bucket", "topic", "prediction_a", "prediction_b", "text"}
assert len(rows) == 36 and required <= set(rows[0])
assert {row["split"] for row in rows} == {"validation"}
print({"data_kind": DATA_KIND, "source": data_source, "rows": len(rows)})

{'data_kind': 'COURSE_FIXTURE', 'source': 'github', 'rows': 36}


## 1) Macro-F1 و95% bootstrap CI

Macro-F1 يعطي كل فئة وزنًا متساويًا. الـbootstrap يعيد أخذ أمثلة **paired** مع الاستبدال ليظهر عدم اليقين في هذه العينة الصغيرة. CI ليست وعدًا بأداء الإنتاج، ولا يعالج رفع `n_boot` نقص البيانات.


In [5]:
def macro_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))


def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000, alpha=0.05, seed=42):
    truth = np.asarray(y_true, dtype=object)
    prediction = np.asarray(y_pred, dtype=object)
    if len(truth) == 0 or len(truth) != len(prediction):
        raise ValueError("paired non-empty arrays are required")
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_boot):
        indexes = rng.integers(0, len(truth), len(truth))
        values.append(metric_fn(truth[indexes], prediction[indexes]))
    low, high = np.percentile(values, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {
        "estimate": metric_fn(truth, prediction),
        "ci_low": float(low),
        "ci_high": float(high),
        "n_boot": n_boot,
    }

y_true = [row["topic"] for row in rows]
interval_a = bootstrap_ci(y_true, [row["prediction_a"] for row in rows], macro_f1)
interval_b = bootstrap_ci(y_true, [row["prediction_b"] for row in rows], macro_f1)
print("COURSE_FIXTURE A", interval_a)
print("COURSE_FIXTURE B", interval_b)


COURSE_FIXTURE A {'estimate': 0.7807469040247678, 'ci_low': 0.6212498015141271, 'ci_high': 0.8982313491237977, 'n_boot': 1000}
COURSE_FIXTURE B {'estimate': 0.7819444444444444, 'ci_low': 0.6169212785382484, 'ci_high': 0.9042122693118788, 'n_boot': 1000}


## 2) مقارنة زوجية عادلة | Paired bootstrap B − A

يجب أن يستخدم الإصداران الأمثلة نفسها في كل resample. إذا شملت CI الصفر، لا نقول إن B أفضل أو أسوأ اتجاهيًا؛ نقول إن العينة لا تدعم هذا الادعاء.


In [6]:
def paired_bootstrap_difference(y_true, prediction_a, prediction_b, metric_fn, n_boot=1000, alpha=0.05, seed=42):
    truth = np.asarray(y_true, dtype=object)
    first = np.asarray(prediction_a, dtype=object)
    second = np.asarray(prediction_b, dtype=object)
    if len(truth) == 0 or not (len(truth) == len(first) == len(second)):
        raise ValueError("three paired non-empty arrays are required")
    rng = np.random.default_rng(seed)
    differences = []
    for _ in range(n_boot):
        indexes = rng.integers(0, len(truth), len(truth))
        differences.append(metric_fn(truth[indexes], second[indexes]) - metric_fn(truth[indexes], first[indexes]))
    low, high = np.percentile(differences, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    observed = metric_fn(truth, second) - metric_fn(truth, first)
    return {
        "difference_b_minus_a": float(observed),
        "ci_low": float(low),
        "ci_high": float(high),
        "supports_directional_claim": bool(low > 0 or high < 0),
    }

paired_result = paired_bootstrap_difference(
    y_true,
    [row["prediction_a"] for row in rows],
    [row["prediction_b"] for row in rows],
    macro_f1,
)
verdict = (
    "The interval excludes zero; report the observed direction with scope and limitations."
    if paired_result["supports_directional_claim"]
    else "The interval includes zero; this fixture does not support a directional superiority claim."
)
print("COURSE_FIXTURE paired B-A", paired_result)
print("VERDICT:", verdict)


COURSE_FIXTURE paired B-A {'difference_b_minus_a': 0.0011975404196766792, 'ci_low': -0.10472316166062473, 'ci_high': 0.09963005211966207, 'supports_directional_claim': False}
VERDICT: The interval includes zero; this fixture does not support a directional superiority claim.


## 3) Sliced evaluation: المتوسط لا يكفي

نقرأ الأداء حسب `language` و`variant` و`length_bucket`. نستخدم حد مراجعة تعليميًا قدره 15 مثالًا؛ وكل شريحة أصغر تحمل `SMALL_SLICE`. لا نخفيها ولا نعمم منها، بل نستخدمها لإنتاج سؤال أو خطة جمع بيانات.


In [7]:
def sliced_report(rows, pred_key, slice_keys, min_slice_size=15, n_boot=500):
    groups = [("ALL", rows)]
    for key in slice_keys:
        for value in sorted({row[key] for row in rows}):
            groups.append((f"{key}={value}", [row for row in rows if row[key] == value]))
    report = []
    for offset, (name, group) in enumerate(groups):
        interval = bootstrap_ci(
            [row["topic"] for row in group],
            [row[pred_key] for row in group],
            macro_f1,
            n_boot=n_boot,
            seed=42 + offset,
        )
        report.append({
            "slice": name,
            "n": len(group),
            "flag": "SMALL_SLICE" if len(group) < min_slice_size else "",
            **interval,
        })
    return report

slice_report = sliced_report(rows, "prediction_b", ["language", "variant", "length_bucket"])
for item in slice_report:
    print(item)


{'slice': 'ALL', 'n': 36, 'flag': '', 'estimate': 0.7819444444444444, 'ci_low': 0.6111751181328388, 'ci_high': 0.8961752409994097, 'n_boot': 500}
{'slice': 'language=ar', 'n': 24, 'flag': '', 'estimate': 0.758288770053476, 'ci_low': 0.5606515151515151, 'ci_high': 0.9265451388888888, 'n_boot': 500}
{'slice': 'language=en', 'n': 12, 'flag': 'SMALL_SLICE', 'estimate': 0.8285714285714286, 'ci_low': 0.49999999999999994, 'ci_high': 1.0, 'n_boot': 500}
{'slice': 'variant=English', 'n': 12, 'flag': 'SMALL_SLICE', 'estimate': 0.8285714285714286, 'ci_low': 0.496540854978355, 'ci_high': 1.0, 'n_boot': 500}
{'slice': 'variant=Gulf', 'n': 12, 'flag': 'SMALL_SLICE', 'estimate': 0.6583333333333333, 'ci_low': 0.29405906593406594, 'ci_high': 0.8951767676767677, 'n_boot': 500}
{'slice': 'variant=MSA', 'n': 12, 'flag': 'SMALL_SLICE', 'estimate': 0.8374999999999999, 'ci_low': 0.5304166666666666, 'ci_high': 1.0, 'n_boot': 500}
{'slice': 'length_bucket=long', 'n': 18, 'flag': '', 'estimate': 0.5259259259259

## 4) Behavioural tests: هل يحترم العقد؟

الـmetric على dataset تجيب “كم؟”. الاختبار السلوكي يجيب “هل ينجح في سلوك محدد مهم؟”. هذه الحالات مأخوذة من `COURSE_FIXTURE`; أضف حالاتك أنت لكل إصلاح قبل اعتماده.


In [8]:
by_id = {row["example_id"]: row for row in rows}
BEHAVIOURAL_CASES = [
    {"case": "Gulf verification code → digital_service", "example_id": "EV-001", "expected": "digital_service"},
    {"case": "MSA upload failure → digital_service", "example_id": "EV-017", "expected": "digital_service"},
    {"case": "Gulf missing health result → health", "example_id": "EV-006", "expected": "health"},
    {"case": "English missing lab result → health", "example_id": "EV-030", "expected": "health"},
    {"case": "MSA bus route → transport", "example_id": "EV-020", "expected": "transport"},
    {"case": "English double charge → permit", "example_id": "EV-035", "expected": "permit"},
]
behavioural_results = []
for case in BEHAVIOURAL_CASES:
    actual = by_id[case["example_id"]]["prediction_b"]
    behavioural_results.append({**case, "actual": actual, "passed": actual == case["expected"]})
behavioural_summary = {
    "passed": sum(row["passed"] for row in behavioural_results),
    "total": len(behavioural_results),
}
behavioural_summary["pass_rate"] = behavioural_summary["passed"] / behavioural_summary["total"]
for row in behavioural_results:
    print(row)
print("COURSE_FIXTURE behavioural", behavioural_summary)


{'case': 'Gulf verification code → digital_service', 'example_id': 'EV-001', 'expected': 'digital_service', 'actual': 'digital_service', 'passed': True}
{'case': 'MSA upload failure → digital_service', 'example_id': 'EV-017', 'expected': 'digital_service', 'actual': 'digital_service', 'passed': True}
{'case': 'Gulf missing health result → health', 'example_id': 'EV-006', 'expected': 'health', 'actual': 'transport', 'passed': False}
{'case': 'English missing lab result → health', 'example_id': 'EV-030', 'expected': 'health', 'actual': 'transport', 'passed': False}
{'case': 'MSA bus route → transport', 'example_id': 'EV-020', 'expected': 'transport', 'actual': 'digital_service', 'passed': False}
{'case': 'English double charge → permit', 'example_id': 'EV-035', 'expected': 'permit', 'actual': 'permit', 'passed': True}
COURSE_FIXTURE behavioural {'passed': 3, 'total': 6, 'pass_rate': 0.5}


## 5) Taxonomy يدوية، لا أوتوماتيكية

اقرأ النص والتوقع والlabel والسياق. اختر tag واحدة أولية، واكتب rationale قصيرًا. القائمة التالية مثال تعليمي على أخطاء B؛ استبدلها بأخطاء validation في مشروعك.

Tags: `label_noise`, `class_confusion`, `dialect_gap`, `negation`, `truncation`, `preprocessing`, `entity_boundary`, `hard_or_ambiguous`.


In [9]:
ALLOWED_TAGS = {
    "label_noise", "class_confusion", "dialect_gap", "negation",
    "truncation", "preprocessing", "entity_boundary", "hard_or_ambiguous",
}
TAGGED_ERRORS = [
    {"example_id": "EV-006", "taxonomy_tag": "dialect_gap", "rationale": "Gulf phrasing for a missing health result confused with transport."},
    {"example_id": "EV-007", "taxonomy_tag": "hard_or_ambiguous", "rationale": "The attachment mention lacks an explicit permit cue."},
    {"example_id": "EV-010", "taxonomy_tag": "dialect_gap", "rationale": "Gulf app wording obscures the prescription-renewal intent."},
    {"example_id": "EV-012", "taxonomy_tag": "dialect_gap", "rationale": "Colloquial location question lacks the formal transport terms."},
    {"example_id": "EV-019", "taxonomy_tag": "class_confusion", "rationale": "Status language overlaps generic digital-service failures."},
    {"example_id": "EV-020", "taxonomy_tag": "class_confusion", "rationale": "Map/application cue dominates the route intent."},
    {"example_id": "EV-030", "taxonomy_tag": "hard_or_ambiguous", "rationale": "Missing-record wording overlaps multiple service domains."},
    {"example_id": "EV-031", "taxonomy_tag": "hard_or_ambiguous", "rationale": "The short request omits the permit noun."},
]

seen = set()
for item in TAGGED_ERRORS:
    assert item["taxonomy_tag"] in ALLOWED_TAGS
    assert item["example_id"] not in seen
    source_row = by_id[item["example_id"]]
    assert source_row["prediction_b"] != source_row["topic"]
    seen.add(item["example_id"])
taxonomy_counts = Counter(item["taxonomy_tag"] for item in TAGGED_ERRORS)
print("COURSE_FIXTURE taxonomy", dict(taxonomy_counts))


COURSE_FIXTURE taxonomy {'dialect_gap': 3, 'hard_or_ambiguous': 3, 'class_confusion': 2}


## 6) من الخطأ إلى خطة إصلاح | Evidence → action

صيغة الإصلاح الجيد: **المشكلة + الدليل + التغيير + اختبار القبول**.

| الأولوية | إصلاح fixture المقترح | دليل البداية | اختبار القبول |
|---:|---|---|---|
| 1 | زيادة وتنقيح أمثلة Gulf للصحة والنقل | `dialect_gap` متكرر | behavioural cases الجديدة + slice CI |
| 2 | إضافة أمثلة contrastive لطلب/تطبيق مقابل تصريح/مسار | `class_confusion` | انخفاض أخطاء الزوج دون إضرار macro-F1 |
| 3 | مراجعة الإرشادات وجمع سياق للأمثلة القصيرة الملتبسة | `hard_or_ambiguous` | agreement review ثم metric جديد |

هذه ليست وصفة إنتاج؛ هي قراءة قابلة للاختبار من fixture. في مشروعك يجب أن تأتي الأولويات من أخطائك أنت.


In [10]:
recommendations = [
    {
        "priority": 1,
        "issue": "Gulf coverage for health and transport",
        "evidence": "dialect_gap tags in validation fixture",
        "change": "collect/review targeted Gulf examples",
        "acceptance_test": "new behavioural cases plus sliced CI",
    },
    {
        "priority": 2,
        "issue": "class confusion around app/status wording",
        "evidence": "EV-019 and EV-020",
        "change": "add contrastive examples and review label guide",
        "acceptance_test": "paired comparison without regression in other slices",
    },
    {
        "priority": 3,
        "issue": "underspecified short requests",
        "evidence": "hard_or_ambiguous tags",
        "change": "request context or abstain when confidence is low",
        "acceptance_test": "ambiguity behavioural suite",
    },
]
print(json.dumps(recommendations, ensure_ascii=False, indent=2))


[
  {
    "priority": 1,
    "issue": "Gulf coverage for health and transport",
    "evidence": "dialect_gap tags in validation fixture",
    "change": "collect/review targeted Gulf examples",
    "acceptance_test": "new behavioural cases plus sliced CI"
  },
  {
    "priority": 2,
    "issue": "class confusion around app/status wording",
    "evidence": "EV-019 and EV-020",
    "change": "add contrastive examples and review label guide",
    "acceptance_test": "paired comparison without regression in other slices"
  },
  {
    "priority": 3,
    "issue": "underspecified short requests",
    "evidence": "hard_or_ambiguous tags",
    "change": "request context or abstain when confidence is low",
    "acceptance_test": "ambiguity behavioural suite"
  }
]


## 7) حفظ التقرير | Save inspectable evidence

ملفات JSON/CSV التالية أدلة وسيطة. انسخ النتائج المهمة إلى `EVALUATION_REPORT.md` و`MODEL_CARD.md` مع وصف البيانات والقيود والقرار.


In [11]:
reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)
evaluation_report = {
    "data_kind": DATA_KIND,
    "split": "validation",
    "rows": len(rows),
    "macro_f1_a": interval_a,
    "macro_f1_b": interval_b,
    "paired_b_minus_a": paired_result,
    "paired_verdict": verdict,
    "behavioural": behavioural_summary,
    "taxonomy_counts": dict(taxonomy_counts),
    "top_fixes": recommendations,
}
(reports_dir / "day3_evaluation_fixture.json").write_text(
    json.dumps(evaluation_report, ensure_ascii=False, indent=2), encoding="utf-8"
)

with (reports_dir / "day3_slice_report.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(slice_report[0]))
    writer.writeheader()
    writer.writerows(slice_report)
with (reports_dir / "day3_error_taxonomy.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(TAGGED_ERRORS[0]))
    writer.writeheader()
    writer.writerows(TAGGED_ERRORS)
print("REPORTS_WRITTEN", sorted(path.name for path in reports_dir.glob("day3_*")))


REPORTS_WRITTEN ['day3_error_taxonomy.csv', 'day3_evaluation_fixture.json', 'day3_slice_report.csv']


## 🟣 مشروعي — تقييم نماذجي الفعلية وتحليل أخطائها (T5)

هذا القسم لا يستخدم fixture. يقرأ مخرجات دفاتري **المنفّذة والمحفوظة في GitHub** (03، 04، 05، 06) من نسخة المستودع، ثم:

1. يحسب مقاييس test المجمّد مع **95% bootstrap CI** ومقارنة زوجية baseline↔Transformer، وشرائح اللغة بأحجامها.
2. يجمع أخطاء **validation** الفعلية من كل المهام (التصنيف، المشاعر، NER، QA، الاسترجاع). وإذا كانت أقل من 5، يضيف أخطاء test **بعد التجميد** بوسم واضح، ولا يستخدمها في أي ضبط.
3. يقترح تصنيفًا لكل خطأ بقاعدة شفافة (`suggest_error_tag`)، **ثم أراجعه بنفسي**: أقرأ كل خطأ، وأصحّح التصنيف في `MY_TAG_OVERRIDES` عند الحاجة، وأجعل `TAXONOMY_REVIEWED_BY_ME = True`.
4. يرتّب ثلاثة إصلاحات من أكثر الفئات تكرارًا، ويربط كل إصلاح بفئته وبمقياس القبول واختبار عدم الرجوع.

> إذا ظهر خطأ «Missing»، فشغّلي الدفاتر 03–06 واحفظي كلًّا منها في GitHub أولًا، ثم أعيدي تشغيل هذا الدفتر من البداية.

In [12]:
# 🟣 مشروعي — تحميل مخرجات دفاتري المنفّذة من نسخة المستودع (المصدر والخلية مسجّلان)
from bayan.project import collect_notebook_results

NOTEBOOK_DIR = REPO_DIR / "notebooks"
project_results = {}
for notebook_name in ["03_text_classification.ipynb", "04_ner_and_qa.ipynb",
                      "05_arabic_nlp.ipynb", "06_semantic_search.ipynb"]:
    project_results.update(collect_notebook_results(NOTEBOOK_DIR / notebook_name))
REQUIRED_RESULTS = ["nb03_classification", "nb04_ner_qa", "nb05_arabic", "nb06_retrieval"]
missing_results = [name for name in REQUIRED_RESULTS if name not in project_results]
if missing_results:
    raise RuntimeError(
        "Missing executed outputs: " + ", ".join(missing_results)
        + ". شغّلي الدفاتر 03–06 واحفظيها في GitHub أولًا، ثم أعيدي تشغيل هذا الدفتر."
    )
for name in REQUIRED_RESULTS:
    print(name, "←", project_results[name]["_provenance"], "| repo_sha:", project_results[name].get("repo_sha"))

nb03_classification ← {'notebook': '03_text_classification.ipynb', 'code_cell': 18, 'execution_count': None} | repo_sha: 5d83b7de5efbb4864d2fd3e53ef0d790bb4f0557
nb04_ner_qa ← {'notebook': '04_ner_and_qa.ipynb', 'code_cell': 20, 'execution_count': 20} | repo_sha: d54ae98a11caa015bb28e4f928fde7c9d45b4a7e
nb05_arabic ← {'notebook': '05_arabic_nlp.ipynb', 'code_cell': 13, 'execution_count': 13} | repo_sha: 2421d4194582f50036cf2cf354227c23ba201b67
nb06_retrieval ← {'notebook': '06_semantic_search.ipynb', 'code_cell': 15, 'execution_count': 15} | repo_sha: 7df70d391bb00cd81b106147d86f274e7e0e70b8


In [13]:
# 🟣 مشروعي — test المجمّد: 95% bootstrap CI + مقارنة زوجية + شرائح اللغة
from bayan.eval_stats import bootstrap_ci as project_bootstrap_ci
from bayan.eval_stats import paired_bootstrap_difference, sliced_metric_report

def observed_macro_f1(truth, prediction):
    labels = sorted(set(truth) | set(prediction))
    return float(f1_score(truth, prediction, labels=labels, average="macro", zero_division=0))

classification_predictions = project_results["nb03_classification"]["predictions"]
test_predictions = [r for r in classification_predictions if r["split"] == "test"]
validation_predictions = [r for r in classification_predictions if r["split"] == "validation"]
classification_eval = {}
for head in ["topic", "sentiment"]:
    truth = [r[f"{head}_gold"] for r in test_predictions]
    baseline_pred = [r[f"{head}_baseline"] for r in test_predictions]
    transformer_pred = [r[f"{head}_transformer"] for r in test_predictions]
    classification_eval[head] = {
        "n": len(test_predictions),
        "baseline": project_bootstrap_ci(truth, baseline_pred, observed_macro_f1, n_boot=2000),
        "transformer": project_bootstrap_ci(truth, transformer_pred, observed_macro_f1, n_boot=2000),
        "paired_transformer_minus_baseline": paired_bootstrap_difference(
            truth, baseline_pred, transformer_pred, observed_macro_f1, n_boot=2000),
        "slices": sliced_metric_report(test_predictions, true_key=f"{head}_gold", pred_key=f"{head}_transformer",
                                       slice_keys=["language"], metric_fn=observed_macro_f1,
                                       min_slice_size=10, n_boot=1000),
    }

def mean_metric(values, _unused):
    return float(np.mean(np.asarray(values, dtype=float)))

test_rankings = [r for r in project_results["nb06_retrieval"]["test_rankings"] if r["relevant_case_ids"]]
hits, reciprocal_ranks = [], []
for r in test_rankings:
    first = next((rank for rank, case in enumerate(r["ranked_case_ids"][:3], 1) if case in set(r["relevant_case_ids"])), None)
    hits.append(float(first is not None))
    reciprocal_ranks.append(1.0 / first if first else 0.0)
retrieval_ci = {
    "n": len(test_rankings),
    "recall@3": project_bootstrap_ci(hits, hits, mean_metric, n_boot=2000),
    "mrr@3": project_bootstrap_ci(reciprocal_ranks, reciprocal_ranks, mean_metric, n_boot=2000),
}
for head, item in classification_eval.items():
    print(head, "n =", item["n"])
    for name in ["baseline", "transformer"]:
        ci = item[name]
        print(f"   {name:11s} macro-F1 = {ci['estimate']:.3f}  95% CI [{ci['ci_low']:.3f}, {ci['ci_high']:.3f}]")
    diff = item["paired_transformer_minus_baseline"]
    print(f"   Transformer − baseline = {diff['difference_b_minus_a']:+.3f}  CI [{diff['ci_low']:+.3f}, {diff['ci_high']:+.3f}]"
          f"  directional claim supported: {diff['supports_directional_claim']}")
    for s in item["slices"]:
        print(f"   slice {s['slice']:12s} n={s['n']} F1={s['estimate']:.3f} CI [{s['ci_low']:.3f}, {s['ci_high']:.3f}] {s['flag']}")
print("retrieval test", {k: (v if isinstance(v, int) else {x: round(y, 3) for x, y in v.items()}) for k, v in retrieval_ci.items()})

topic n = 8
   baseline    macro-F1 = 0.733  95% CI [0.314, 1.000]
   transformer macro-F1 = 0.867  95% CI [0.523, 1.000]
   Transformer − baseline = +0.133  CI [-0.386, +0.675]  directional claim supported: False
   slice ALL          n=8 F1=0.867 CI [0.500, 1.000] SMALL_SLICE
   slice language=ar  n=4 F1=0.667 CI [0.333, 1.000] SMALL_SLICE
   slice language=en  n=4 F1=1.000 CI [1.000, 1.000] SMALL_SLICE
sentiment n = 8
   baseline    macro-F1 = 1.000  95% CI [1.000, 1.000]
   transformer macro-F1 = 0.356  95% CI [0.095, 0.733]
   Transformer − baseline = -0.644  CI [-0.905, -0.267]  directional claim supported: True
   slice ALL          n=8 F1=0.356 CI [0.095, 0.733] SMALL_SLICE
   slice language=ar  n=4 F1=0.167 CI [0.000, 0.429] SMALL_SLICE
   slice language=en  n=4 F1=0.600 CI [0.200, 1.000] SMALL_SLICE
retrieval test {'n': 6, 'recall@3': {'estimate': 1.0, 'bootstrap_mean': 1.0, 'ci_low': 1.0, 'ci_high': 1.0}, 'mrr@3': {'estimate': 0.667, 'bootstrap_mean': 0.667, 'ci_low': 0.5, '

In [14]:
# 🟣 مشروعي — تجميع أخطاء نماذجي الفعلية واقتراح تصنيف لكل خطأ
from bayan.project import qa_exact_and_f1, suggest_error_tag, taxonomy_summary

def classification_errors(prediction_rows, split_name):
    found = []
    for r in prediction_rows:
        for head in ["topic", "sentiment"]:
            if r[f"{head}_transformer"] != r[f"{head}_gold"]:
                found.append({"example_id": f'{r["example_id"]}:{head}', "task": f"{head}_classification",
                              "split": split_name, "language": r["language"], "text": r["text"],
                              "gold": r[f"{head}_gold"], "prediction": r[f"{head}_transformer"]})
    return found

error_pool = classification_errors(validation_predictions, "validation")
for index, e in enumerate(project_results["nb04_ner_qa"]["ner"]["errors"]):
    if e["split"] == "validation":
        error_pool.append({"example_id": f"NER-val-{index}", "task": "ner", "split": "validation",
                           "language": e["language"], "text": " ".join(e["tokens"]),
                           "gold": " ".join(e["gold"]), "prediction": " ".join(e["prediction"])})
for item in project_results["nb04_ner_qa"]["qa"]["predictions"]["validation"]:
    if qa_exact_and_f1(item["prediction"], item["gold"])[0] < 1.0:
        error_pool.append({"example_id": item["id"], "task": "qa", "split": "validation", "language": item["language"],
                           "text": item["question"], "gold": item["gold"], "prediction": item["prediction"]})
for e in project_results["nb06_retrieval"]["validation_errors"]:
    error_pool.append({**e, "task": "retrieval"})

POST_FREEZE_USED = len(error_pool) < 5
if POST_FREEZE_USED:
    # Too few validation errors to read patterns: add frozen-test errors, labelled, for analysis only (never for tuning).
    test_errors = classification_errors(test_predictions, "test_post_freeze")
    comparison = project_results["nb05_arabic"]["dialect_comparison"]["results"][0]
    for row, predicted in zip(project_results["nb05_arabic"]["gulf_test_rows"], comparison["gulf_test_predictions"]):
        if predicted != row["topic"]:
            test_errors.append({"example_id": row["record_id"], "task": "topic_classification_gulf", "split": "test_post_freeze",
                                "language": "ar", "variant": row["variant"], "text": row["text"],
                                "gold": row["topic"], "prediction": predicted})
    for index, e in enumerate(project_results["nb04_ner_qa"]["ner"]["errors"]):
        if e["split"] == "test":
            test_errors.append({"example_id": f"NER-test-{index}", "task": "ner", "split": "test_post_freeze",
                                "language": e["language"], "text": " ".join(e["tokens"]),
                                "gold": " ".join(e["gold"]), "prediction": " ".join(e["prediction"])})
    for item in project_results["nb04_ner_qa"]["qa"]["predictions"]["test"]:
        if qa_exact_and_f1(item["prediction"], item["gold"])[0] < 1.0:
            test_errors.append({"example_id": item["id"], "task": "qa", "split": "test_post_freeze", "language": item["language"],
                                "text": item["question"], "gold": item["gold"], "prediction": item["prediction"]})
    error_pool += test_errors
print("errors collected:", len(error_pool), "| post-freeze test errors added:", POST_FREEZE_USED)

errors collected: 6 | post-freeze test errors added: False


In [15]:
# ✍️ مراجعتي اليدوية — اقرئي كل خطأ مطبوع أدناه. إذا خالفتِ التصنيف المقترح، اكتبي التصنيف الصحيح وسببك هنا
#    (المفتاح هو example_id كما يظهر في الجدول)، ثم اجعلي TAXONOMY_REVIEWED_BY_ME = True وأعيدي التشغيل.
MY_TAG_OVERRIDES = {
    # "F-017:topic": ("dialect_gap", "سببي بكلماتي"),
}
TAXONOMY_REVIEWED_BY_ME = False

tagged_errors = []
for e in error_pool:
    suggestion = suggest_error_tag(e)
    tag, rationale = MY_TAG_OVERRIDES.get(e["example_id"], (suggestion["taxonomy_tag"], suggestion["rationale"]))
    source = ("manual_override" if e["example_id"] in MY_TAG_OVERRIDES
              else "rule_suggestion_reviewed" if TAXONOMY_REVIEWED_BY_ME else "rule_suggestion_unreviewed")
    tagged_errors.append({**e, "taxonomy_tag": tag, "rationale": rationale, "tag_source": source})
assert all(e["taxonomy_tag"] in ALLOWED_TAGS for e in tagged_errors)
project_taxonomy = taxonomy_summary(tagged_errors) if tagged_errors else []
for e in tagged_errors:
    print(f'{e["split"]:16s} {e["task"]:26s} {e["example_id"]:16s} | {str(e["text"])[:45]:45s} | gold={e["gold"]} | pred={e["prediction"]} → {e["taxonomy_tag"]} ({e["tag_source"]})')
print()
for row in project_taxonomy:
    print(row["taxonomy_tag"], row["count"], row["example_ids"], "|", row["example"])

validation       sentiment_classification   D-027:sentiment  | أين أجد نتيجة الموعد الطبي                    | gold=neutral | pred=negative → class_confusion (rule_suggestion_unreviewed)
validation       sentiment_classification   D-037:sentiment  | تغير وقت وصول الحافلة                         | gold=neutral | pred=negative → class_confusion (rule_suggestion_unreviewed)
validation       sentiment_classification   D-038:sentiment  | The bus arrival time has changed              | gold=neutral | pred=negative → class_confusion (rule_suggestion_unreviewed)
validation       ner                        NER-val-0        | Case BAYAN-302 belongs to the transport servi | gold=O B-REF_NUM O O O B-SERVICE I-SERVICE | pred=O B-REF_NUM O O O O O → entity_boundary (rule_suggestion_unreviewed)
validation       qa                         Q-007            | كيف يمكن تقديم بلاغ النقل؟                    | gold=عبر التطبيق أو مركز الاتصال | pred=بلاغ النقل عبر التطبيق → entity_boundary (rule_suggestion_

In [16]:
# 🟣 مشروعي — ثلاثة إصلاحات مرتّبة؛ كل إصلاح مربوط بفئة خطأ ومقياس قبول واختبار عدم رجوع
FIX_LIBRARY = {
    "dialect_gap": ("إضافة أمثلة خليجية مراجَعة لكل فئة في train (≥5 لكل فئة) وإعادة التدريب", "Gulf-slice macro-F1 (دفتر 05) + شريحة language=ar", "متوسطة", "حالات خليجية ثابتة في behavioural tests + عدم انخفاض MSA"),
    "class_confusion": ("إضافة أمثلة contrastive للزوج المختلط ومراجعة دليل التسميات", "macro-F1 للفئتين المختلطتين", "منخفضة", "paired bootstrap دون تراجع في بقية الشرائح"),
    "negation": ("إضافة أزواج نفي/إثبات (لم/ما/not) إلى train", "directional tests للنفي", "منخفضة", "حالات نفي ثابتة في tests/"),
    "hard_or_ambiguous": ("طلب سياق أو الامتناع عند انخفاض الثقة، مع عتبة تُضبط على validation", "no-answer accuracy ونسبة الامتناع", "منخفضة", "حالات غامضة ثابتة يجب ألا تُجاب"),
    "entity_boundary": ("توسيع أمثلة الكيانات متعددة الكلمات ومراجعة محاذاة I- للكلمات اللاحقة", "strict entity F1 لكل نوع", "متوسطة", "اختبار حدود الكيان في tests/"),
    "truncation": ("رفع MAX_LENGTH للمرشح التالي أو استخدام stride", "truncation rate + F1 للنصوص الطويلة", "منخفضة", "length audit بلا قطع"),
    "preprocessing": ("مراجعة profile المعالجة وإضافة golden test للحالة", "golden tests", "منخفضة", "tests/test_my_arabic_golden.py"),
    "label_noise": ("مراجعة التسمية بمصدر ثانٍ وتصحيح البيانات بإصدار جديد", "agreement بعد المراجعة", "منخفضة", "data hash جديد + قرار في DECISIONS.md"),
}
ranked_tags = [row["taxonomy_tag"] for row in project_taxonomy]
for known_risk in ["dialect_gap", "hard_or_ambiguous", "class_confusion"]:
    if len(ranked_tags) >= 3:
        break
    if known_risk not in ranked_tags:
        ranked_tags.append(known_risk)
prioritized_fixes = []
for priority, tag in enumerate(ranked_tags[:3], start=1):
    action, metric, cost, regression = FIX_LIBRARY[tag]
    observed = next((row for row in project_taxonomy if row["taxonomy_tag"] == tag), None)
    prioritized_fixes.append({
        "priority": priority, "taxonomy_tag": tag,
        "evidence": (f"{observed['count']} observed: " + ", ".join(observed["example_ids"][:4])) if observed
                    else "not observed in this run; known risk from slices/data card",
        "action": action, "expected_metric": metric, "cost": cost, "regression_test": regression,
    })
print(json.dumps(prioritized_fixes, ensure_ascii=False, indent=2))

[
  {
    "priority": 1,
    "taxonomy_tag": "class_confusion",
    "evidence": "3 observed: D-027:sentiment, D-037:sentiment, D-038:sentiment",
    "action": "إضافة أمثلة contrastive للزوج المختلط ومراجعة دليل التسميات",
    "expected_metric": "macro-F1 للفئتين المختلطتين",
    "cost": "منخفضة",
    "regression_test": "paired bootstrap دون تراجع في بقية الشرائح"
  },
  {
    "priority": 2,
    "taxonomy_tag": "entity_boundary",
    "evidence": "3 observed: NER-val-0, Q-007, Q-008",
    "action": "توسيع أمثلة الكيانات متعددة الكلمات ومراجعة محاذاة I- للكلمات اللاحقة",
    "expected_metric": "strict entity F1 لكل نوع",
    "cost": "متوسطة",
    "regression_test": "اختبار حدود الكيان في tests/"
  },
  {
    "priority": 3,
    "taxonomy_tag": "dialect_gap",
    "evidence": "not observed in this run; known risk from slices/data card",
    "action": "إضافة أمثلة خليجية مراجَعة لكل فئة في train (≥5 لكل فئة) وإعادة التدريب",
    "expected_metric": "Gulf-slice macro-F1 (دفتر 05) + شريحة langua

In [17]:
# 🟣 مشروعي — حفظ الدليل القابل للتتبع
emit_result("nb07_evaluation", {
    **PROVENANCE,
    "sources": {name: project_results[name]["_provenance"] for name in REQUIRED_RESULTS},
    "source_repo_shas": {name: project_results[name].get("repo_sha") for name in REQUIRED_RESULTS},
    "classification": classification_eval,
    "retrieval_ci": retrieval_ci,
    "qa": project_results["nb04_ner_qa"]["qa"]["metrics"],
    "ner": project_results["nb04_ner_qa"]["ner"]["by_split"],
    "behavioural": project_results["nb03_classification"]["behavioural"],
    "error_pool_size": len(error_pool),
    "post_freeze_test_errors_used": POST_FREEZE_USED,
    "errors": tagged_errors,
    "taxonomy": project_taxonomy,
    "taxonomy_reviewed_by_student": TAXONOMY_REVIEWED_BY_ME,
    "prioritized_fixes": prioritized_fixes,
})

BAYAN_RESULT::nb07_evaluation::{"behavioural":{"by_perturbation":{"diacritic":{"pass_rate":0.75,"passed":6,"total":8},"double_spaces":{"pass_rate":1.0,"passed":16,"total":16},"lowercase":{"pass_rate":1.0,"passed":8,"total":8},"tatweel":{"pass_rate":1.0,"passed":8,"total":8},"trailing_punctuation":{"pass_rate":0.875,"passed":14,"total":16}},"overall":{"pass_rate":0.9285714285714286,"passed":52,"total":56}},"classification":{"sentiment":{"baseline":{"bootstrap_mean":1.0,"ci_high":1.0,"ci_low":1.0,"estimate":1.0},"n":8,"paired_transformer_minus_baseline":{"ci_high":-0.2666666666666666,"ci_low":-0.9047619047619048,"difference_b_minus_a":-0.6444444444444444,"supports_directional_claim":true},"slices":[{"bootstrap_mean":0.34957688052688046,"ci_high":0.7333333333333334,"ci_low":0.09523809523809523,"estimate":0.35555555555555557,"flag":"SMALL_SLICE","n":8,"slice":"ALL"},{"bootstrap_mean":0.16579365079365077,"ci_high":0.42857142857142855,"ci_low":0.0,"estimate":0.16666666666666666,"flag":"SMALL

'BAYAN_RESULT::nb07_evaluation::{"behavioural":{"by_perturbation":{"diacritic":{"pass_rate":0.75,"passed":6,"total":8},"double_spaces":{"pass_rate":1.0,"passed":16,"total":16},"lowercase":{"pass_rate":1.0,"passed":8,"total":8},"tatweel":{"pass_rate":1.0,"passed":8,"total":8},"trailing_punctuation":{"pass_rate":0.875,"passed":14,"total":16}},"overall":{"pass_rate":0.9285714285714286,"passed":52,"total":56}},"classification":{"sentiment":{"baseline":{"bootstrap_mean":1.0,"ci_high":1.0,"ci_low":1.0,"estimate":1.0},"n":8,"paired_transformer_minus_baseline":{"ci_high":-0.2666666666666666,"ci_low":-0.9047619047619048,"difference_b_minus_a":-0.6444444444444444,"supports_directional_claim":true},"slices":[{"bootstrap_mean":0.34957688052688046,"ci_high":0.7333333333333334,"ci_low":0.09523809523809523,"estimate":0.35555555555555557,"flag":"SMALL_SLICE","n":8,"slice":"ALL"},{"bootstrap_mean":0.16579365079365077,"ci_high":0.42857142857142855,"ci_low":0.0,"estimate":0.16666666666666666,"flag":"SMAL

## بوابة Core | Core gate

علامة النجاح تتحقق من provenance، CI، المقارنة الزوجية، الشرائح، taxonomy، والتقارير. النجاح البرمجي لا يعوض كتابة تفسيرك في تقرير المشروع.


In [18]:
core_checks = {
    "fixture_disclosed": evaluation_report["data_kind"] == "COURSE_FIXTURE",
    "validation_only": evaluation_report["split"] == "validation",
    "confidence_intervals": interval_a["ci_low"] <= interval_a["estimate"] <= interval_a["ci_high"],
    "paired_comparison": "supports_directional_claim" in paired_result,
    "slices_present": len(slice_report) >= 8,
    "small_slice_flags_present": any(row["flag"] == "SMALL_SLICE" for row in slice_report),
    "behavioural_tests": behavioural_summary["total"] >= 5,
    "manual_taxonomy": len(TAGGED_ERRORS) >= 5 and bool(taxonomy_counts),
    "three_ranked_fixes": [row["priority"] for row in recommendations] == [1, 2, 3],
    "project_three_fixes": [row["priority"] for row in prioritized_fixes] == [1, 2, 3],
    "project_ci_computed": all("ci_low" in item["transformer"] for item in classification_eval.values()),
    "reports_written": all((reports_dir / name).exists() for name in [
        "day3_evaluation_fixture.json", "day3_slice_report.csv", "day3_error_taxonomy.csv"
    ]),
}
assert all(core_checks.values()), core_checks
print(core_checks)
print("DAY3_NOTEBOOK7_CORE=PASS")

{'fixture_disclosed': True, 'validation_only': True, 'confidence_intervals': True, 'paired_comparison': True, 'slices_present': True, 'small_slice_flags_present': True, 'behavioural_tests': True, 'manual_taxonomy': True, 'three_ranked_fixes': True, 'project_three_fixes': True, 'project_ci_computed': True, 'reports_written': True}
DAY3_NOTEBOOK7_CORE=PASS


## بعد المختبر | After the lab

1. استبدل fixture بتنبؤات مشروعك على validation.
2. أكمل [قالب تقرير التقييم](../templates/EVALUATION_REPORT_TEMPLATE.md) و[بطاقة النموذج](../templates/MODEL_CARD_TEMPLATE.md).
3. اكتب تفسير CI والشرائح وثلاثة إصلاحات في `DECISIONS.md`.
4. استخدم commit: `docs: add evaluation and error analysis`.
5. ارجع إلى [Gate C](../day-03/04-labs-checkpoint.md) وأكمل checklist قبل مغادرة اليوم الثالث.
